# Week 3, Lab 4 — Crew + tools


In [ ]:
WEEK = 'Week 3'
LAB = 'Lab 4 — crew tools'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


In [ ]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

llm = LLM(
    model=f"openai/{cfg['model']}",
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
)
print("CrewAI LLM ->", cfg)


In [ ]:
from crewai.tools import BaseTool

class LookupTool(BaseTool):
    name: str = "lookup_fact"
    description: str = "Look up a local fact about agentic AI topics."
    def _run(self, topic: str) -> str:
        return lookup_fact(topic)

class CalcTool(BaseTool):
    name: str = "calculator"
    description: str = "Evaluate arithmetic like '12*8+3'."
    def _run(self, expression: str) -> str:
        return calculator(expression)

researcher = Agent(
    role="Researcher",
    goal="Use lookup_fact for topic facts.",
    backstory="Librarian of the local KB.",
    llm=llm,
    tools=[LookupTool()],
)
analyst = Agent(
    role="Analyst",
    goal="Use calculator for any math.",
    backstory="Likes numbers.",
    llm=llm,
    tools=[CalcTool()],
)
t1 = Task(description="What is LangGraph? Use the lookup tool.", expected_output="1-2 sentences grounded in the tool.", agent=researcher)
t2 = Task(description="Compute 45*12+30 with the calculator and include it in a closing sentence.", expected_output="Short recap including the number.", agent=analyst)
print(Crew(agents=[researcher, analyst], tasks=[t1, t2], process=Process.sequential).kickoff())


**Next:** 3-agent research → draft → review.
